In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.4 The Pseudoinverse, Rank Deficiency, and Regularization

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Volume II — Orthogonality and Least Squares",
    number="2.4",
    title="The Pseudoinverse, Rank Deficiency, and Regularization",
    blurb="What to do when the columns are dependent, the answer is not unique, "
    "or the problem is ill-posed — ending in a deblurring that float64 "
    "destroys by sixteen orders of magnitude and one well-chosen parameter "
    "repairs.",
    difficulty="advanced",
    estimate="105–135 min",
)

## Notebook overview

Every method in [§2.3](least-squares-four-ways.ipynb) assumed $A$ has
independent columns. That assumption is doing more work than it looks: it is
what makes $A^{\top}A$ invertible, what makes $R$ nonsingular, and what makes
the least-squares answer *unique*. Drop it and the normal equations become
singular, $QR$ has a zero on its diagonal, and the question "what is the
solution?" stops having an answer, because there are infinitely many.

The **pseudoinverse** $A^{+}$ is the object that survives. It exists for every
matrix — square or not, full rank or not — it agrees with $A^{-1}$ whenever the
inverse exists, and when the answer is not unique it makes a specific choice:
the solution of smallest norm. It is built from the SVD in one line, which is
the third time in this course that the SVD has been the factorization that
keeps working after the others have stopped.

Then the harder half. A problem can fail to have a usable answer not because
the matrix is singular but because it is *ill-posed*: the forward map genuinely
destroys information, so no algorithm can return it. The worked example is
deblurring a 1-D signal. Blurring it is a matrix multiply. Undoing it is
`np.linalg.solve`, and that call returns an answer wrong by a factor of
$2\times10^{15}$ — not because `solve` is bad, but because the blur operator's
smallest singular values are $10^{-18}$ and inverting them amplifies the noise
in the last decimal place into the dominant term. The repair is
**regularization**: deliberately answering a slightly different question whose
answer is stable. One parameter, chosen from the data without knowing the
truth, brings the error back to 3.5%.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Golub and Van Loan {cite}`golub2013` §5.5 for the pseudoinverse;
> Hansen {cite}`hansen2010` is the book for everything after Exercise 4 —
> filter factors, the discrete Picard condition, the L-curve — and is
> unusually readable. Penrose's original four conditions are
> {cite}`penrose1955`; the $\ell^1$ material is Tibshirani
> {cite}`tibshirani1996` and Hastie, Tibshirani and Friedman
> {cite}`hastie2009` Chapter 3.

## Theory in brief

### The pseudoinverse

Let $A$ be $m\times n$ with rank $r$ and economy SVD
$A = U\Sigma V^{\top}$, where $\Sigma = \operatorname{diag}(\sigma_1 \ge \dots
\ge \sigma_r > 0, 0, \dots, 0)$. The **Moore–Penrose pseudoinverse** is

```{math}
:label: eq-pinv-svd
A^{+} = V\Sigma^{+}U^{\top}, \qquad
\Sigma^{+} = \operatorname{diag}\!\left(
  \tfrac{1}{\sigma_1}, \dots, \tfrac{1}{\sigma_r}, 0, \dots, 0\right).
```

The rule is: **invert what you can, and zero what you cannot**. The zeros are
not an approximation or a convenience; they are what makes $A^{+}$ unique.

Penrose's characterisation is that $A^{+}$ is the *only* matrix satisfying all
four of

```{math}
:label: eq-pinv-mp
AA^{+}A = A, \qquad A^{+}AA^{+} = A^{+}, \qquad
(AA^{+})^{\top} = AA^{+}, \qquad (A^{+}A)^{\top} = A^{+}A .
```

The first two say $A^{+}$ is a *generalized* inverse (each matrix undoes the
other on the part of the space where that is possible); the last two say the
two products are **symmetric**, hence orthogonal projectors. Indeed

```{math}
:label: eq-pinv-projectors
AA^{+} = P_{\mathcal{C}(A)}, \qquad A^{+}A = P_{\mathcal{R}(A)},
```

the orthogonal projectors onto the column space and the row space, exactly the
objects [§1.4](../01-matrices/four-subspaces.ipynb) built by hand. Since a
projector's trace is its rank, $\operatorname{tr}(AA^{+}) = r$ — a cheap and
strict check.

When $A$ is square and nonsingular, {eq}`eq-pinv-svd` gives $A^{+} = A^{-1}$.
When $A$ is tall with independent columns it gives the least-squares solution
operator $(A^{\top}A)^{-1}A^{\top}$ of [§2.1](projections-normal-equations.ipynb).
The general statement covering both:

```{math}
:label: eq-pinv-minnorm
A^{+}\mathbf{b} = \operatorname*{arg\,min}
  \left\{\|\mathbf{x}\|_2 \;:\;
  \mathbf{x} \in \operatorname*{arg\,min}_{\mathbf{z}}
  \|A\mathbf{z} - \mathbf{b}\|_2 \right\}.
```

Among all vectors that minimise the residual, $A^{+}\mathbf{b}$ is the shortest.
Both minimisations matter: the outer one because the residual may not be zero,
the inner one because the minimiser may not be unique.

### Truncation, and why an exact pseudoinverse is not always what one wants

$A^{+}$ inverts every nonzero singular value, and "nonzero" is a decision no
floating-point computation can make. Worse, the inversion is *unstable* where
$\sigma_i$ is small: a component of $\mathbf{b}$ along $\mathbf{u}_i$ is
amplified by $1/\sigma_i$. The **truncated SVD** solution simply stops early,

```{math}
:label: eq-pinv-tsvd
\mathbf{x}_k = \sum_{i=1}^{k}
  \frac{\mathbf{u}_i^{\top}\mathbf{b}}{\sigma_i}\,\mathbf{v}_i ,
```

discarding the directions whose inversion would do more harm than good.

### Tikhonov regularization

The smoother alternative penalises the solution's size rather than truncating:

```{math}
:label: eq-pinv-tikhonov
\mathbf{x}_\lambda = \operatorname*{arg\,min}_{\mathbf{x}}
  \left\{ \|A\mathbf{x} - \mathbf{b}\|_2^2
         + \lambda\|\mathbf{x}\|_2^2 \right\}
  = (A^{\top}\!A + \lambda I)^{-1}A^{\top}\mathbf{b} ,
  \qquad \lambda > 0 .
```

This goes by **ridge regression** in statistics and **weight decay** in machine
learning; all three are {eq}`eq-pinv-tikhonov`. Adding $\lambda I$ shifts every
eigenvalue of $A^{\top}A$ up by $\lambda$, so the matrix is positive definite
for any $\lambda > 0$ *whatever the rank of $A$*, and the solution always
exists and is always unique.

Substituting the SVD reveals what it really does:

```{math}
:label: eq-pinv-filter
\mathbf{x}_\lambda = \sum_{i=1}^{n} f_i(\lambda)\,
  \frac{\mathbf{u}_i^{\top}\mathbf{b}}{\sigma_i}\,\mathbf{v}_i ,
\qquad
f_i(\lambda) = \frac{\sigma_i^2}{\sigma_i^2 + \lambda} .
```

The $f_i$ are the **filter factors**. Each is a smooth switch: $f_i \approx 1$
where $\sigma_i \gg \sqrt{\lambda}$ and $f_i \approx \sigma_i^2/\lambda \approx
0$ where $\sigma_i \ll \sqrt{\lambda}$. Truncation {eq}`eq-pinv-tsvd` is the
same formula with the hard filter $f_i = 1$ for $i \le k$ and $0$ after, so the
two methods differ only in the shape of one switch. The sum
$\sum_i f_i(\lambda)$ is the **effective number of degrees of freedom**: how
many directions the fit is actually using.

Forming $A^{\top}A$ in {eq}`eq-pinv-tikhonov` squares the condition number, the
precise sin of [§2.3](least-squares-four-ways.ipynb). The cure is the same as
there — never form it. Note that

```{math}
:label: eq-pinv-stacked
\|A\mathbf{x} - \mathbf{b}\|_2^2 + \lambda\|\mathbf{x}\|_2^2
= \left\| \begin{bmatrix} A \\ \sqrt{\lambda}\,I \end{bmatrix}\mathbf{x}
         - \begin{bmatrix} \mathbf{b} \\ \mathbf{0} \end{bmatrix} \right\|_2^2 ,
```

so Tikhonov is an ordinary least-squares problem on a stacked
$(m+n)\times n$ matrix, solvable by $QR$ with no $A^{\top}A$ anywhere.

### Ill-posed, not merely ill-conditioned

A discretised integral equation — a blur, a heat equation run backwards, a
tomographic projection — has singular values that decay *exponentially* to
zero, with no gap and no natural cut-off. The useful diagnostic is the
**discrete Picard condition**: the solution is recoverable only where the data
coefficients $|\mathbf{u}_i^{\top}\mathbf{b}|$ decay *faster* than the
$\sigma_i$, so that the ratios in {eq}`eq-pinv-tsvd` stay bounded,

```{math}
:label: eq-pinv-picard
\frac{|\mathbf{u}_i^{\top}\mathbf{b}|}{\sigma_i} \;\lesssim\; \text{const}.
```

Noise puts a floor under $|\mathbf{u}_i^{\top}\mathbf{b}|$: past the index
where $\sigma_i$ drops below that floor the ratio explodes, and every term
beyond it is amplified noise. That crossover is where to stop, and it can be
read off the data alone.

### Choosing $\lambda$ from the data

The **L-curve** {cite}`hansen2010` plots $\log\|\mathbf{x}_\lambda\|$ against
$\log\|A\mathbf{x}_\lambda - \mathbf{b}\|$ as $\lambda$ sweeps. Large $\lambda$
gives a small solution and a large residual (over-smoothed); small $\lambda$
gives the reverse (noise-dominated). The curve is L-shaped and the **corner**,
the point of maximum curvature, is the compromise. The **discrepancy
principle** is the alternative when the noise level $\delta$ is known: choose
the largest $\lambda$ with $\|A\mathbf{x}_\lambda - \mathbf{b}\| \le \delta$,
on the reasoning that fitting the data more closely than its own accuracy is
fitting noise.

### Changing the penalty

Nothing forces the penalty to be $\|\mathbf{x}\|_2^2$. Replacing it by the
$1$-norm gives

```{math}
:label: eq-pinv-lasso
\mathbf{x}_\lambda^{\ell^1} = \operatorname*{arg\,min}_{\mathbf{x}}
  \left\{ \tfrac{1}{2}\|A\mathbf{x} - \mathbf{b}\|_2^2
         + \lambda\|\mathbf{x}\|_1 \right\},
```

the **lasso** {cite}`tibshirani1996`. It has no closed form and no SVD
expansion, and its solutions are **sparse**: entries are set to *exactly* zero,
not merely made small. The geometric reason is the shape of the unit ball,
which [§0.3](../00-machine/vectors-norms-inner-products.ipynb) drew — the
$\ell^1$ ball has corners, and corners lie on the coordinate axes.

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import qr

from ecp import draw, validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed
# The deconvolution's noise gets its own generator so that its realisation — and
# therefore every number quoted in Exercises 5 to 7 — does not depend on how many
# draws the earlier exercises happened to make from `rng`.
rng_blur = np.random.default_rng(2)

EPS = np.finfo(float).eps
np.set_printoptions(precision=5, suppress=False, linewidth=110)

# The rank-3 matrix of the Prologue: its third column is the sum of the first
# two, so no algorithm can invert it and every method in 2.3 fails on it.
A_DEP = np.array([[2.0, 1.0, 3.0, 0.0],
                  [3.0, 4.0, 7.0, 0.0],
                  [1.0, 2.0, 3.0, -1.0],
                  [0.0, 1.0, 1.0, 2.0]])


def relative_error(x, x_true):
    """Relative 2-norm error of a computed vector against the exact one."""
    return float(np.linalg.norm(x - x_true) / np.linalg.norm(x_true))

## Exercise 1 — Building $A^{+}$, and the four conditions that pin it down

{eq}`eq-pinv-svd` is a three-line construction: take the SVD, reciprocate the
singular values above a cut-off, zero the rest, and reassemble with $U$ and $V$
swapped. What is worth dwelling on is the *uniqueness*. Many matrices $G$
satisfy $AGA = A$ — that alone is a weak condition with a large solution set.
All four of {eq}`eq-pinv-mp` together are satisfied by exactly one matrix, and
{eq}`eq-pinv-svd` is it. That is why the four are worth checking one at a time
rather than accepting the construction on trust.

The test matrix is the $4\times4$ matrix of the Prologue,

$$
A = \begin{bmatrix}
2 & 1 & 3 & 0\\ 3 & 4 & 7 & 0\\ 1 & 2 & 3 & -1\\ 0 & 1 & 1 & 2
\end{bmatrix},
$$

available as `A_DEP` above. Its third column is the sum of the first two, so
its rank is 3 and its fourth singular value is exactly zero in exact
arithmetic. It is singular, so `np.linalg.inv` is meaningless on it and
`np.linalg.solve` will return nonsense or raise.

**Part a)** Take the full SVD with `U, s, Vt = np.linalg.svd(A_DEP)` and report
the four singular values. Fix the cut-off `cut = 1e-10 * s[0]`, set
`r = int((s > cut).sum())`, and confirm $r = 3$.

**Part b)** Build $A^{+}$ from {eq}`eq-pinv-svd` as a two-line construction:
form `s_plus = np.zeros_like(s)`, set `s_plus[:r] = 1.0 / s[:r]`, then
`A_plus = (Vt.T * s_plus) @ U.T` — the broadcast `Vt.T * s_plus` scales column
$i$ of $V$ by $1/\sigma_i$, which is $V\Sigma^{+}$ without materialising the
diagonal matrix. Compare against `np.linalg.pinv(A_DEP)` entrywise; the two
should agree to $10^{-14}$.

**Write this one yourself** — the implementation is the lesson.

**Part c)** Check all four Moore–Penrose conditions {eq}`eq-pinv-mp` by
computing the maximum absolute entry of $AA^{+}A - A$, of $A^{+}AA^{+} - A^{+}$,
of $(AA^{+})^{\top} - AA^{+}$, and of $(A^{+}A)^{\top} - A^{+}A$. All four come
out below $4\times10^{-15}$.

**Part d)** Confirm {eq}`eq-pinv-projectors`. Form `P_col = A_DEP @ A_plus` and
`P_row = A_plus @ A_DEP`, and check that each is idempotent
($P^2 = P$ to $10^{-14}$), that each has trace 3 to $10^{-12}$, and that
`P_col` equals `U[:, :r] @ U[:, :r].T` to $10^{-14}$ — the projector built from
the first three left singular vectors, exactly as in
[§1.4](../01-matrices/four-subspaces.ipynb).

**Part e)** Report $\max_{ij}|(AA^{+} - I)_{ij}|$, which comes out $0.420$.
This is the number that says $A^{+}$ is *not* an inverse: $AA^{+}$ is a
projector of rank 3 in a 4-dimensional space, and no rank-3 matrix can be the
identity.

In [ ]:
# (solution hidden on the public site)


### Validation 1

The four conditions are checked separately rather than as a single aggregate,
because they say four different things: the first two make $A^{+}$ a
generalized inverse, the last two make the two products *orthogonal*
projectors. The trace check is the strictest of the lot — a projector's trace
is its rank, an integer, so any drift shows immediately.

In [ ]:
for name, val in mp.items():
    validate.check(val < 4e-15, f"Moore-Penrose condition {name} (Eq. 2)",
                   f"max |residual entry| = {val:.3e}")
validate.close(
    A_plus, np.linalg.pinv(A_DEP),
    "the hand-built V Sigma^+ U^T agrees with np.linalg.pinv (Eq. 1)",
    rtol=0.0, atol=1e-14,
)
validate.close(
    np.array([np.trace(P_col), np.trace(P_row)]), np.array([3.0, 3.0]),
    "tr(A A+) = tr(A+ A) = rank(A) = 3 (Eq. 3): a projector's trace is its rank",
    rtol=0.0, atol=1e-12,
)
validate.close(
    P_col, U[:, :r] @ U[:, :r].T,
    "A A+ is the orthogonal projector onto the column space (Eq. 3)",
    rtol=0.0, atol=1e-14,
)
validate.check(
    np.abs(P_col - np.eye(4)).max() > 0.4,
    "and A A+ is emphatically not the identity",
    f"max |A A+ - I| = {np.abs(P_col - np.eye(4)).max():.4f}: a rank-3 projector "
    "cannot be the 4-by-4 identity, so A+ is a generalized inverse and not an "
    "inverse",
)

## Exercise 2 — The pseudoinverse is discontinuous, and `rcond` is the reason

Matrix inversion is continuous wherever it is defined: perturb a nonsingular
$A$ slightly and $A^{-1}$ moves slightly. The pseudoinverse has no such
guarantee. $A^{+}$ is a **discontinuous** function of $A$ at every matrix where
the rank can change, and rank-deficient matrices are exactly those matrices.

The mechanism is visible in {eq}`eq-pinv-svd`. A perturbation that lifts
$\sigma_4$ from $0$ to $\delta$ does not perturb $A^{+}$ by $O(\delta)$; it
*adds a whole new term* of size $1/\delta$. The map $\sigma \mapsto 1/\sigma$
has no limit at $0$, so as $\delta \to 0^{+}$ the pseudoinverse of the
perturbed matrix diverges, while the pseudoinverse of the limit matrix is
perfectly finite. There is no repair available in exact arithmetic. What there
is instead is a *decision*, and `rcond` is where it is made: every singular
value below `rcond * s[0]` is declared zero and dropped.

The Prologue asked what happens to a rank-3 matrix perturbed by $10^{-12}$, and
left the question open. This exercise answers it.

**Part a)** For each $\delta \in \{0, 10^{-14}, 10^{-12}, 10^{-10},
10^{-8}\}$, form `A_d = A_DEP.copy()` and add $\delta$ to the single entry
`A_d[3, 2]`. Report the smallest singular value from
`np.linalg.svd(A_d, compute_uv=False)`, the rank from
`np.linalg.matrix_rank(A_d)`, and the spectral norm
`np.linalg.norm(np.linalg.pinv(A_d), 2)` using the **default** `rcond`.

**Part b)** Identify the jump. Between $\delta = 10^{-14}$ and
$\delta = 10^{-12}$ the norm of the pseudoinverse rises from $0.860$ to
$5.35\times10^{12}$, a factor of $6\times10^{12}$, in response to a change of
$10^{-12}$ in one entry of `A_DEP`, whose largest entry is 7. Confirm the jump
is at least $10^{12}$, and confirm it coincides with `matrix_rank` changing
from 3 to 4.

**Part c)** Explain the threshold quantitatively. `np.linalg.pinv` drops
singular values below `rcond * s[0]` with `rcond = 1e-15` by default, so the
cut-off here is $1.0\times10^{-14}$. Report $\sigma_4$ for each $\delta$ and
confirm the jump happens exactly where $\sigma_4$ crosses that cut-off: at
$\delta = 10^{-14}$, $\sigma_4 = 1.7\times10^{-15}$ is below it and gets
dropped; at $\delta = 10^{-12}$, $\sigma_4 = 1.9\times10^{-13}$ is above it and
gets inverted.

**Part d)** Show that the decision is yours to make. Repeat Part a) with
`np.linalg.pinv(A_d, rcond=1e-10)` and confirm $\|A^{+}\|_2$ is *identical* to
eight significant figures across $\delta = 0, 10^{-14}, 10^{-12}, 10^{-10}$,
and only moves at $\delta = 10^{-8}$. Choosing `rcond` is choosing what counts
as noise — and there is no default that is right for every problem, which is
the entire content of this exercise.

In [ ]:
# (solution hidden on the public site)


### Validation 2

The check is not that the pseudoinverse is large — it is that a $10^{-12}$
perturbation moves it by $10^{12}$, which is the definition of a discontinuity
reached numerically. The second half confirms the discontinuity is *removable
by decision*: with `rcond` pinned, the same perturbations change nothing.

In [ ]:
validate.check(
    jump > 1e12,
    "a 1e-12 perturbation changes |A+|_2 by a factor above 1e12 (Eq. 1)",
    f"|A+|_2 goes from {norm_default[1]:.4f} to {norm_default[2]:.4e}: the "
    "pseudoinverse is discontinuous where the rank can change",
)
validate.check(
    ranks == [3, 3, 4, 4, 4],
    "and the jump coincides exactly with matrix_rank changing from 3 to 4",
    f"ranks {ranks} for deltas {[f'{d:.0e}' for d in deltas]}",
)
validate.check(
    sig4[1] < cutoff_default < sig4[2],
    "the threshold is where sigma_4 crosses rcond * sigma_1, not anywhere else",
    f"sigma_4 = {sig4[1]:.3e} < {cutoff_default:.3e} < {sig4[2]:.3e}",
)
validate.close(
    np.array(norm_pinned[:4]), np.full(4, norm_pinned[0]),
    "with rcond=1e-10 pinned, the first four perturbations change nothing",
    rtol=1e-8, atol=0.0,
)
validate.check(
    norm_pinned[4] > 1e8 * norm_pinned[0],
    "and rcond=1e-10 moves only once sigma_4 crosses ITS cut-off",
    f"|A+|_2 = {norm_pinned[4]:.3e} at delta = 1e-8, where "
    f"sigma_4 = {sig4[4]:.3e} exceeds 1e-10 * sigma_1 = {1e-10 * s[0]:.3e}",
)

## Exercise 3 — The minimum-norm solution, and what the normal equations do instead

When $A$ is $m\times n$ with $m < n$ and full row rank, $A\mathbf{x} =
\mathbf{b}$ has infinitely many exact solutions: a particular one plus anything
in the null space, which here has dimension $n - m$. Every one of them fits the
data perfectly, so "smallest residual" cannot choose between them.
{eq}`eq-pinv-minnorm` chooses by a second criterion, the norm, and
[§1.4](../01-matrices/four-subspaces.ipynb) already showed why that criterion
picks out exactly one vector: split any solution as $\mathbf{x} =
\mathbf{x}_{\text{row}} + \mathbf{x}_{\text{null}}$, note the two parts are
orthogonal, and Pythagoras gives
$\|\mathbf{x}\|^2 = \|\mathbf{x}_{\text{row}}\|^2 +
\|\mathbf{x}_{\text{null}}\|^2$. Adding null-space content can only lengthen
the vector, so the shortest solution is the one with none: the row-space
component, which is $A^{+}\mathbf{b}$.

In [ ]:
# (solution hidden on the public site)


The system for this exercise is the $3\times6$ matrix

$$
A_{\text{wide}} = \begin{bmatrix}
1 & 2 & 0 & 1 & -1 & 3\\
0 & 1 & 1 & -2 & 2 & 1\\
2 & 0 & -1 & 3 & 1 & 0
\end{bmatrix},
\qquad
\mathbf{b} = \begin{bmatrix} 6\\ 2\\ 5\end{bmatrix},
$$

which has rank 3 and therefore a 3-dimensional null space and a
3-parameter family of exact solutions.

**Part a)** Build `A_wide` and `b_wide` as written above, confirm
`np.linalg.matrix_rank(A_wide)` is 3, and compute
`x_mn = np.linalg.pinv(A_wide) @ b_wide`. Report it and its norm, which comes
out $2.0217$, and confirm the residual $\|A\mathbf{x} - \mathbf{b}\|_2$ is
below $10^{-13}$ — it is an exact solution, not a least-squares compromise.

**Part b)** Confirm `np.linalg.lstsq(A_wide, b_wide, rcond=None)[0]` returns
the *same* vector to $10^{-13}$. This is worth knowing: `lstsq` is documented
as returning the minimum-norm least-squares solution, and on an underdetermined
system that is the minimum-norm exact solution.

**Part c)** Extract a null-space basis from the full SVD:
`Uw, sw, Vtw = np.linalg.svd(A_wide)` and `N = Vtw[3:].T`, of shape `(6, 3)`.
Confirm $\|A_{\text{wide}}N\|_{\max} < 10^{-13}$ and that
$\|N^{\top}\mathbf{x}_{\text{mn}}\|_{\max} < 10^{-13}$, so the minimum-norm
solution has no null-space component at all.

**Part d)** Test {eq}`eq-pinv-minnorm` by brute force. Draw
`coef = rng.standard_normal((10_000, 3))` and form the $10^4$ alternative
solutions `alts = x_mn + coef @ N.T`. Confirm every one has residual below
$10^{-13}$ (they are all genuine solutions) and that every one has norm at
least $\|\mathbf{x}_{\text{mn}}\|$. Also confirm Pythagoras holds to $10^{-13}$:
$\|\mathbf{x}_{\text{alt}}\|^2 = \|\mathbf{x}_{\text{mn}}\|^2 +
\|\text{coef}\cdot N^{\top}\|^2$ for each of the $10^4$.

**Part e)** Now the cautionary half. Form the $6\times6$ Gram matrix
$A^{\top}A$, which is singular in exact arithmetic because
$\operatorname{rank}(A^{\top}A) = \operatorname{rank}(A) = 3 < 6$, and solve
the normal equations anyway with
`np.linalg.solve(A_wide.T @ A_wide, A_wide.T @ b_wide)`. Report
`np.linalg.cond` and `np.linalg.det` of the Gram matrix, and report whether the
call raises.

It does **not** raise. Floating-point LU never meets an exactly zero pivot, so
it returns a vector, and that vector solves the system to $10^{-14}$: a
perfectly valid member of the 3-parameter solution family. It is simply not the
minimum-norm member. Confirm that its norm strictly exceeds
$\|\mathbf{x}_{\text{mn}}\| = 2.0217$, and verify the mechanism exactly by
checking Pythagoras once more: $\|\mathbf{x}_{\text{ne}}\|^2 -
\|\mathbf{x}_{\text{mn}}\|^2 = \|N^{\top}\mathbf{x}_{\text{ne}}\|^2$ to
$10^{-12}$, so the whole of the excess is null-space content that the
minimum-norm solution does not carry.

*How much* longer is deliberately not stated here, and neither is $\kappa$ or
the determinant: which member of the family LU lands on depends on its pivot
sequence, which depends on the BLAS, so the numbers this cell prints differ
between machines. That is the sharper version of the warning. The normal
equations on a rank-deficient system do not merely return the wrong answer;
they return a machine-dependent one, with no exception, no warning, and a
residual that looks perfect.

In [ ]:
# (solution hidden on the public site)


### Validation 3

The minimum-norm property is checked against $10^4$ genuine competitors rather
than asserted, and Pythagoras is checked on every one of them, because the
orthogonal split is the *reason* the property holds. The last check records
what the normal equations returned: not an error, and not the right answer.

In [ ]:
validate.close(
    A_wide @ x_mn, b_wide,
    "A+ b solves the underdetermined system exactly (Eq. 4)",
    rtol=0.0, atol=1e-13,
)
validate.close(
    x_lstsq, x_mn,
    "np.linalg.lstsq returns the same minimum-norm solution",
    rtol=0.0, atol=1e-13,
)
validate.check(
    np.abs(N.T @ x_mn).max() < 1e-13,
    "A+ b lies entirely in the row space: no null-space component (Eq. 3)",
    f"|N^T x_mn|_max = {np.abs(N.T @ x_mn).max():.3e} for the 3-dimensional "
    "null space",
)
validate.check(
    (alt_res < 1e-13).all() and (alt_norms >= np.linalg.norm(x_mn) - 1e-12).all(),
    "and it is the shortest of 10,000 exact solutions (Eq. 4)",
    f"every alternative solves to {alt_res.max():.2e}; the shortest found was "
    f"{alt_norms.min():.6f} against |A+ b| = {np.linalg.norm(x_mn):.6f}",
)
validate.close(
    pythag, np.zeros_like(pythag),
    "Pythagoras holds for all 10,000: |x|^2 = |x_row|^2 + |x_null|^2",
    rtol=0.0, atol=1e-12,
)
validate.check(
    np.linalg.norm(A_wide @ x_ne - b_wide) < 1e-14 * np.linalg.norm(b_wide)
    and np.linalg.norm(x_ne) > np.linalg.norm(x_mn) * (1 + 1e-6),
    "the normal equations returned a VALID but longer solution, silently",
    f"residual {np.linalg.norm(A_wide @ x_ne - b_wide):.2e} with norm "
    f"{np.linalg.norm(x_ne):.4f} against the minimum {np.linalg.norm(x_mn):.4f}: "
    "no exception was raised, and HOW much longer is a LAPACK pivot-order "
    "detail that differs between machines, so only 'strictly longer' is gated",
)
validate.close(
    np.array([ne_excess]), np.array([ne_null]),
    "and the entire excess is null-space content, by Pythagoras (Eq. 4)",
    rtol=0.0, atol=1e-12,
)

## Exercise 4 — Filter factors: truncation and ridge are one formula

{eq}`eq-pinv-filter` says something stronger than "here is another method". It
says truncation and Tikhonov are the *same computation* differing only in a
sequence of numbers $f_1, \dots, f_n$ multiplying the SVD expansion. Truncation
uses a step, $f_i = \mathbb{1}[i \le k]$; Tikhonov uses the smooth switch
$\sigma_i^2/(\sigma_i^2 + \lambda)$, which is $\tfrac{1}{2}$ exactly where
$\sigma_i = \sqrt{\lambda}$. Seeing them this way makes the whole family
visible at once, and makes $\sqrt{\lambda}$ readable directly off a plot of the
singular values.

The test matrix is `la.random_with_condition(60, 12, 1e2, rng)`, of shape
$(60, 12)$ with condition number exactly $10^{2}$ and singular values in
geometric progression from 1 to $0.01$; the right-hand side is
`b_r = A_r @ x_r + 0.01 * rng.standard_normal(60)` for
`x_r = rng.standard_normal(12)`. It is deliberately well conditioned, so that
every discrepancy below is the algebra and not the arithmetic. Part f) then
repeats one comparison on `la.random_with_condition(60, 12, 1e6, rng)`, where
that stops being true.

**Part a)** Verify the three routes to $\mathbf{x}_\lambda$ agree. For
$\lambda \in \{10^{-6}, 10^{-8}, 10^{-10}, 10^{-12}\}$ compute it three ways:
the matrix form {eq}`eq-pinv-tikhonov` with
`np.linalg.solve(A_r.T @ A_r + lam * np.eye(12), A_r.T @ b_r)`; the filter form
{eq}`eq-pinv-filter` as `Vr.T @ ((sr**2 / (sr**2 + lam)) * (Ur.T @ b_r) / sr)`
from the economy SVD `Ur, sr, Vr = np.linalg.svd(A_r, full_matrices=False)`;
and the stacked form {eq}`eq-pinv-stacked` by building the $(72, 12)$ matrix
`np.vstack([A_r, np.sqrt(lam) * np.eye(12)])`, the length-72 right-hand side
`np.concatenate([b_r, np.zeros(12)])`, and solving it with
`scipy.linalg.qr(..., mode="economic")` followed by a triangular solve. All
three agree to $10^{-12}$ here.

**Part b)** Confirm the $\lambda \to 0$ limit *and its rate*. Compute
$\|\mathbf{x}_\lambda - \mathbf{x}_{\text{LS}}\|_\infty$ against
`x_ls = np.linalg.lstsq(A_r, b_r, rcond=None)[0]` for the same four $\lambda$,
and confirm the deviation falls by a factor of 100 each time — the bias
introduced by ridge is $O(\lambda)$, not $O(\sqrt{\lambda})$ or
$O(\lambda^2)$. Check the ratio of consecutive deviations is $100$ to within
1.5%. The four $\lambda$ are chosen below $\sigma_{\min}^2 = 10^{-4}$, since
the linear regime is where $\lambda$ is small compared to the eigenvalues it
is being added to.

**Part c)** Confirm the $\lambda \to \infty$ limit *and its rate*. For
$\lambda = 10^{3}, 10^{6}, 10^{9}$ check that
$\|\mathbf{x}_\lambda\| \to \|A^{\top}\mathbf{b}\|/\lambda$, which follows from
{eq}`eq-pinv-tikhonov` because $\lambda I$ dominates $A^{\top}A$. Agreement
should reach 6 significant figures by $\lambda = 10^{6}$.

**Part d)** Compute the effective degrees of freedom
$\sum_i f_i(\lambda)$ for $\lambda$ ranging over `np.logspace(-14, 2, 25)`,
and confirm it decreases monotonically from 12 (all directions used) toward 0
(none), and that it equals 12 to within $10^{-6}$ at $\lambda = 10^{-14}$.

**Part e)** Plot the filter factors against $\sigma_i$ on log-log axes for
$\lambda \in \{10^{-2}, 10^{-4}, 10^{-6}\}$, with the truncation step at
$k = 6$ drawn for comparison, and mark $\sqrt{\lambda}$ on the $\sigma$ axis
for each $\lambda$. Confirm numerically that $f_i = \tfrac{1}{2}$ to
$10^{-12}$ wherever $\sigma_i = \sqrt{\lambda}$, by evaluating the filter
function at $\sigma = \sqrt{\lambda}$ directly.

**Part f)** Now show where the textbook formula stops being usable. Build
`A_ill = la.random_with_condition(60, 12, 1e6, rng)` with
`b_ill = A_ill @ x_ill + 0.01 * rng.standard_normal(60)` for
`x_ill = rng.standard_normal(12)`, and repeat the matrix-versus-filter and
stacked-versus-filter comparison of Part a) at
$\lambda \in \{10^{-2}, 10^{-4}, 10^{-6}, 10^{-8}\}$. The two forms are
algebraically identical, but the matrix form disagrees with the filter form by
$2\times10^{-7}$ at $\lambda = 10^{-8}$ while the stacked form disagrees by
$7\times10^{-12}$: a factor of 30,000. Confirm that the matrix form's error
stays under $\kappa^2\varepsilon = 2.2\times10^{-4}$ and the stacked form's
under $\kappa\varepsilon = 2.2\times10^{-10}$, the two rates
[§2.3](least-squares-four-ways.ipynb) measured. Forming $A^{\top}A$ costs the
same square here as it did there, and {eq}`eq-pinv-stacked` avoids it for the
same reason $QR$ did.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

The three routes are checked against each other rather than against a
reference, since none of them is more "correct" than the others in exact
arithmetic. The limits are checked with their *rates*, which is much stricter
than checking a limit: a wrong formula can easily reach the right limit by the
wrong route.

In [ ]:
validate.check(
    max(agree_mf) < 5e-12 and max(agree_sf) < 5e-12,
    "the matrix, filter, and stacked-QR forms all agree (Eqs. 6, 7, 8)",
    f"worst matrix-vs-filter {max(agree_mf):.2e}, worst stacked-vs-filter "
    f"{max(agree_sf):.2e}, over lambda from 1e-6 to 1e-12 at kappa = 100",
)
validate.close(
    np.array(ratios), np.full(len(ratios), 100.0),
    "the ridge bias is O(lambda): a 100-fold drop in lambda drops it 100-fold",
    # O(sqrt(lambda)) would give ratios of 10 and O(lambda^2) of 10^4, so 10%
    # still separates the candidate rates by orders of magnitude.
    rtol=1e-1, atol=0.0,
)
validate.close(
    np.array(nrm_big[1:]), np.array(pred_big[1:]),
    "and as lambda -> infinity, |x_lam| -> |A^T b| / lambda (Eq. 6)",
    rtol=1e-5, atol=0.0,
)
validate.check(
    abs(dof_grid[0] - 12.0) < 1e-6 and np.all(np.diff(dof_grid) < 0),
    "the effective dof falls monotonically from 12 to 0 as lambda grows (Eq. 7)",
    f"{dof_grid[0]:.9f} at lambda = 1e-14 down to {dof_grid[-1]:.3e} at "
    "lambda = 1e2, strictly decreasing throughout",
)
validate.close(
    half, np.full(3, 0.5),
    "and f = 1/2 exactly where sigma = sqrt(lambda), the filter's midpoint",
    rtol=0.0, atol=1e-12,
)
validate.check(
    ill_mf[-1] > 1e2 * ill_sf[-1],
    "at kappa = 1e6 the matrix form is orders worse than the stacked form",
    f"{ill_mf[-1]:.2e} against {ill_sf[-1]:.2e} at lambda = 1e-8, a factor of "
    f"{ill_mf[-1]/ill_sf[-1]:.0f}: forming A^T A squares the condition number "
    "here exactly as it did in 2.3",
)
validate.check(
    max(ill_mf) < kap_ill**2 * EPS and max(ill_sf) < kap_ill * EPS,
    "and each form stays inside its own predicted rate",
    f"matrix {max(ill_mf):.2e} < kappa^2 eps = {kap_ill**2 * EPS:.2e}; "
    f"stacked {max(ill_sf):.2e} < kappa eps = {kap_ill * EPS:.2e}",
)

## Exercise 5 — An ill-posed problem, and why `solve` cannot survive it

Everything so far has been *ill-conditioned*: hard, but with a finite $\kappa$
and a definite number of digits lost. An **ill-posed** problem is different in
kind. Discretising a smoothing integral operator gives a matrix whose singular
values decay exponentially with no floor, so $\kappa$ is not a large number but
a meaningless one, and the smallest singular values encode information the
forward map genuinely erased.

The example is 1-D Gaussian blur on $n = 200$ equispaced points of $[0,1]$,
available as `la.gaussian_blur(200, 0.03)`. Row $i$ of that matrix samples

$$
K(t_i, t_j) = \frac{1}{w\sqrt{2\pi}}\,
  \exp\!\left(-\frac{(t_i - t_j)^2}{2w^2}\right), \qquad w = 0.03,
$$

on the midpoint grid $t_i = (i + \tfrac{1}{2})/200$ and multiplies by the
quadrature weight $h = 1/200$, so $A\mathbf{x}$ is the discretised convolution
and every row sums to 1. The matrix is symmetric and Toeplitz. The test signal
is

$$
x_j = \exp\!\left(-\tfrac{1}{2}\left(\tfrac{t_j - 0.22}{0.05}\right)^{2}\right)
      + 0.8\cdot\mathbb{1}\!\left[0.55 < t_j < 0.72\right],
$$

a smooth bump beside a sharp box, chosen so that one signal exercises both the
regime regularization handles well and the regime it handles badly. The data is
$\mathbf{b} = A\mathbf{x}_{\text{true}} + \mathbf{e}$ with $\mathbf{e}$ scaled
to exactly $0.1\%$ of $\|A\mathbf{x}_{\text{true}}\|$.

**Part a)** Build `A_blur = la.gaussian_blur(200, 0.03)`, the grid
`t = (np.arange(200) + 0.5) / 200`, the signal `x_true` as written above, and
`b_clean = A_blur @ x_true`. Draw the noise as
`e = rng_blur.standard_normal(200)`, rescale it with
`e *= 1e-3 * np.linalg.norm(b_clean) / np.linalg.norm(e)`, and set
`b_blur = b_clean + e`. Confirm $\|\mathbf{e}\|/\|\mathbf{b}_{\text{clean}}\| =
10^{-3}$ to $10^{-12}$.

**Part b)** Take `Ub, sb, Vtb = np.linalg.svd(A_blur)` and report how many
singular values exceed $\varepsilon\sigma_1$: about 100 of 200. The rest are
below the level at which `float64` can distinguish them from zero, so
`np.linalg.cond` returns a number with no meaning. The *exact* count is not a
reproducible quantity and should not be treated as one — the values either side
of the threshold are themselves rounding error, so a different LAPACK driver
returns a different count for the same matrix. Confirm only that the count
falls between 60 and 140, that `np.linalg.cond(A_blur)` exceeds
$1/\varepsilon$, and report $\sigma_{40}/\sigma_1$, which *is* reproducible at
$1.5\times10^{-3}$ — the decay is exponential, not algebraic.

**Part c)** Solve it naively with `np.linalg.solve(A_blur, b_blur)` and report
the relative error against `x_true`, which comes out near $3\times10^{15}$, and
the norm of the returned vector against $\|\mathbf{x}_{\text{true}}\| = 6.28$.
Then do the same on the **noiseless** data `b_clean`: the error is still $629$.
This is the fact worth pausing on. With no measurement noise at all, the
rounding error already present in `b_clean` at the $10^{-16}$ level is enough
to destroy the answer, because $1/\sigma_{200} = 8.8\times10^{16}$.

**Part d)** Test the discrete Picard condition {eq}`eq-pinv-picard`. Compute
the ratios $|\mathbf{u}_i^{\top}\mathbf{b}|/\sigma_i$ for both `b_clean` and
`b_blur`. Report them across the whole spectrum, but *check* them only on the
indices with $\sigma_i > 10^{-10}\sigma_1$, where both the singular values and
the data coefficients are genuine numbers rather than rounding error: the clean
ratios stay below 5 there while the noisy ratios exceed $10^{5}$. Across the
full resolvable spectrum the separation reaches twelve orders. Report the noise
floor
on $|\mathbf{u}_i^{\top}\mathbf{b}|$ as `np.median(coef_noisy[120:])` and confirm
it is around $2\times10^{-4}$, the same order as
$\|\mathbf{e}\|/\sqrt{200} = 4.0\times10^{-4}$.

**Part e)** Plot the singular values on a log axis together with
$|\mathbf{u}_i^{\top}\mathbf{b}|$ for the clean and noisy data, all against
$i$. The crossover — where the noisy coefficients level off onto their floor
while $\sigma_i$ keeps falling — is where the solution stops being recoverable,
and it is visible without knowing $\mathbf{x}_{\text{true}}$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

The naive failure is checked on both the noisy and the noiseless data, because
the second is the stronger statement: the problem is not that the measurement
was imperfect, it is that inverting $\sigma \approx 10^{-18}$ turns the last
bit of any `float64` vector into the dominant term. The Picard check is the
constructive half, and uses only $\mathbf{b}$ and $A$ — never
$\mathbf{x}_{\text{true}}$.

In [ ]:
validate.close(
    np.array([noise_frac]), np.array([1e-3]),
    "the noise is exactly 0.1% of the clean data in the 2-norm",
    rtol=1e-12, atol=0.0,
)
validate.check(
    60 < n_meaningful < 140 and np.linalg.cond(A_blur) > 1 / EPS,
    "roughly half the 200 singular values fall below eps * sigma_1",
    f"{n_meaningful} of 200 above the threshold on this machine (the exact "
    f"count is a LAPACK-driver detail, since the values either side of it ARE "
    f"rounding error), and cond = {np.linalg.cond(A_blur):.2e} exceeds "
    f"1/eps = {1/EPS:.2e}, so kappa is not a measurable quantity here",
)
validate.close(
    np.array([sb[39] / sb[0]]), np.array([1.52e-3]),
    "and the decay is exponential: sigma_40 / sigma_1 = 1.5e-3",
    rtol=1e-2, atol=0.0,
)
validate.check(
    relative_error(x_naive, x_true) > 1e12,
    "np.linalg.solve on the noisy data is wrong by more than 12 orders",
    f"relative error {relative_error(x_naive, x_true):.3e}, returning a vector "
    f"of norm {np.linalg.norm(x_naive):.3e} for a signal of norm "
    f"{np.linalg.norm(x_true):.3f}",
)
validate.check(
    relative_error(x_naive_clean, x_true) > 10.0,
    "and it fails on NOISELESS data too: float64 rounding is already enough",
    f"relative error {relative_error(x_naive_clean, x_true):.3e} with no "
    f"measurement noise at all, because 1/sigma_200 = {1/sb[-1]:.2e}",
)
validate.check(
    ratio_clean[solid].max() < 5.0,
    "the noiseless data satisfies the discrete Picard condition (Eq. 9)",
    f"max |u^T b| / sigma = {ratio_clean[solid].max():.4f} over the "
    f"{int(solid.sum())} indices with sigma > 1e-10 * sigma_1",
)
validate.check(
    ratio_noisy[solid].max() > 1e5,
    "the noisy data violates it, over that same range of genuine numbers",
    f"max ratio {ratio_noisy[solid].max():.3e} against "
    f"{ratio_clean[solid].max():.3f} clean: the noise floor {floor:.2e} on "
    f"|u^T b| stops the decay while sigma keeps falling. Extended to every "
    f"index above eps * sigma_1 the noisy ratio reaches "
    f"{ratio_noisy[ok].max():.2e}, but that far tail is rounding error and is "
    "reported rather than gated",
)

## Exercise 6 — The repair, and the bias it is bought with

Regularization does not solve the problem posed. It solves a nearby problem
whose answer is stable, and the gap between the two is a **bias** that no
amount of data removes. That trade is the whole subject, so it is worth
measuring both halves rather than celebrating the first.

The signal of Exercise 5 was built with two regions on purpose. On
$t < 0.45$ it is a smooth Gaussian bump, exactly the kind of function a penalty
on $\|\mathbf{x}\|_2$ treats gently. On $0.50 < t < 0.80$ it is a box with two
jump discontinuities, which the same penalty rounds off, because a sharp edge
needs large high-frequency components and those are precisely what
{eq}`eq-pinv-filter` suppresses. One signal, two verdicts.

**Part a)** Sweep $\lambda$ over `np.logspace(-12, 1, 261)`, computing
$\mathbf{x}_\lambda$ by the filter form {eq}`eq-pinv-filter` from `Ub, sb, Vtb`
— never the matrix form, whose $A^{\top}A$ would square an already meaningless
condition number. Record `relative_error(x_lam, x_true)` for each, and report
the best $\lambda$ and the error there. The minimum is near
$\lambda = 2\times10^{-5}$ with a relative error of $0.136$.

**Part b)** Report the improvement factor over the naive solve of Exercise 5,
which exceeds $10^{16}$, and confirm it is at least $10^{6}$.

**Part c)** Split the error by region. Using the masks `t < 0.45` and
`(t > 0.50) & (t < 0.80)`, report the relative error of the best
$\mathbf{x}_\lambda$ separately on each. The smooth region comes out at
$0.013$ and the box region at $0.182$, a factor of 14. Confirm the smooth
region is below $0.06$ and that the box region is at least 3 times worse. The
aggregate $0.136$ is not a uniform 13.6% error: it is 1.3% where the signal is
smooth and 18% where it is not, and quoting only the aggregate would hide
which of those two the method is responsible for.

**Part d)** Sweep the truncation parameter $k$ from 1 to 200 using
{eq}`eq-pinv-tsvd`, and report the best $k$ and its error. It lands at
$k = 39$ with an error within 1% of the best Tikhonov error — the two filters
do equally well, as {eq}`eq-pinv-filter` suggests they should. Confirm the best
$k$ falls where $\sigma_k$ is comparable to the noise level $10^{-3}\sigma_1$:
$\sigma_{39}/\sigma_1 = 2.1\times10^{-3}$, so the truncation stops precisely
where the data stops carrying signal, which is what Exercise 5 predicted from
the Picard plot alone.

**Part e)** Plot the reconstruction at three values of $\lambda$ — one far too
small ($10^{-12}$), the best one, and one far too large ($10^{-1}$) — against
`x_true` and against the blurred data `b_blur`, on one figure. Confirm from the
numbers that the too-small $\lambda$ has $\|\mathbf{x}_\lambda\| >
10\,\|\mathbf{x}_{\text{true}}\|$ (noise amplification: it comes out 73 times
larger) while the too-large one has $\|\mathbf{x}_\lambda\| <
\|\mathbf{x}_{\text{true}}\|$ (over-smoothing).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

The headline improvement is checked, but so is the bias: a notebook that
reported only the $10^{16}$ gain would be selling the method rather than
teaching it. The regional split is the honest form of the result.

In [ ]:
validate.check(
    gain > 1e6,
    "regularization beats the naive solve by more than six orders (Eq. 7)",
    f"relative error {err_best:.4f} at lambda = {lam_best:.3e} against "
    f"{relative_error(x_naive, x_true):.3e} naive: a factor of {gain:.3e}",
)
validate.check(
    err_smooth < 0.06,
    "on the smooth part of the signal the reconstruction is good",
    f"relative error {err_smooth:.4f} for t < 0.45",
)
validate.check(
    err_box > 3.0 * err_smooth,
    "but the box's edges are rounded off: that bias is what was bought",
    f"{err_box:.4f} on 0.50 < t < 0.80 against {err_smooth:.4f} on the smooth "
    f"region, a factor of {err_box/err_smooth:.2f}; a penalty on |x| suppresses "
    "exactly the high-frequency components a jump needs",
)
validate.close(
    np.array([err_k.min()]), np.array([err_best]),
    "truncation at its best k does as well as Tikhonov at its best lambda",
    rtol=0.06, atol=0.0,
)
validate.check(
    sb[k_best - 1] / sb[0] < 1e-2 and sb[k_best - 1] / sb[0] > 1e-4,
    "and the best truncation stops where sigma_k meets the noise level",
    f"sigma_{k_best} / sigma_1 = {sb[k_best-1]/sb[0]:.3e} against a noise level "
    "of 1e-3: the Picard plot of Exercise 5 located this without x_true",
)
validate.check(
    np.linalg.norm(x_small) > 10 * np.linalg.norm(x_true)
    and np.linalg.norm(x_large) < np.linalg.norm(x_true),
    "and the two failure modes are visible in the norm alone",
    f"|x| = {np.linalg.norm(x_small):.4f} at lambda = 1e-12 (noise amplified, "
    f"{np.linalg.norm(x_small)/np.linalg.norm(x_true):.0f} times too large) and "
    f"{np.linalg.norm(x_large):.4f} at lambda = 1e-1 (over-smoothed), against "
    f"|x_true| = {np.linalg.norm(x_true):.4f}",
)

## Exercise 7 — Choosing $\lambda$ without the answer key

Exercise 6 chose $\lambda$ by minimising the error against
$\mathbf{x}_{\text{true}}$, which is cheating: if $\mathbf{x}_{\text{true}}$
were available there would be nothing to compute. That sweep produced an
*oracle* $\lambda$, useful only as a yardstick. A usable method must read
$\lambda$ off $A$ and $\mathbf{b}$ alone.

Two such methods are standard. The **L-curve** plots
$\log\|\mathbf{x}_\lambda\|$ against
$\log\|A\mathbf{x}_\lambda - \mathbf{b}\|$; the curve turns sharply where
further reduction of the residual starts costing a great deal of solution norm,
and that corner is the estimate. Its curvature, for a curve given
parametrically as $(u(\lambda), v(\lambda))$, is

```{math}
:label: eq-pinv-curvature
\kappa_{\text{L}} = \frac{u'v'' - v'u''}{\left(u'^2 + v'^2\right)^{3/2}} ,
```

computed here by central differences with `np.gradient`. The **discrepancy
principle** instead uses a known noise level $\delta = \|\mathbf{e}\|$ and
picks the $\lambda$ whose residual matches it, on the argument that driving the
residual below the noise means fitting the noise.

The honest test of either is not whether it recovers the oracle $\lambda$ — it
will not — but whether the $\lambda$ it does return gives an error close to the
oracle's.

**Part a)** For each $\lambda$ in the sweep of Exercise 6, compute
$\|A\mathbf{x}_\lambda - \mathbf{b}\|_2$ and $\|\mathbf{x}_\lambda\|_2$ with
`np.linalg.norm`, take `np.log` of both, and plot the L-curve. Confirm with
`np.diff` that the residual increases monotonically with $\lambda$ and the
solution norm decreases monotonically: the curve is traversed in one
direction, which is what makes a corner meaningful.

**Part b)** Locate the corner. With `u = np.log(res)` and `v = np.log(sol)`,
apply `np.gradient` twice to each, evaluate {eq}`eq-pinv-curvature`, and take
`np.nanargmax`. Report the corner's $\lambda$, and the relative error of
$\mathbf{x}_\lambda$ there against `x_true`.

**Part c)** Compare against the oracle. Report the ratio of the corner's error
to the oracle's error, and separately the gap in $\log_{10}\lambda$. The corner
lands $1.4$ decades below the oracle $\lambda$ and costs $17\%$ more error.
Confirm the error ratio is below $1.3$.

**Part d)** Apply the discrepancy principle. With $\delta = \|\mathbf{e}\|_2$
known, find the sweep index with `np.argmin(np.abs(res_sweep - delta))`, which
minimises $\bigl|\|A\mathbf{x}_\lambda - \mathbf{b}\|_2 - \delta\bigr|$; report its
$\lambda$ and error, and confirm its residual matches $\delta$ to $12\%$ — the
$\lambda$ grid steps by a factor of $1.12$, so the nearest grid point can miss
the exact level by several per cent — and its
error ratio is below $1.3$ as well. It lands $0.6$ decades *above* the oracle
and costs only $1.2\%$.

The two estimates therefore disagree with each other about $\lambda$ by two
full decades while disagreeing about the error by 16%. That is the practical
content of this exercise: the error surface is flat near its minimum, so a
parameter-choice rule does not have to find $\lambda$ accurately to be useful,
and reporting how well it recovered $\lambda$ would badly misstate how well it
performed. Confirm the two $\lambda$ differ by more than one decade while both
error ratios stay under $1.3$.

**Part e)** Plot the L-curve on log-log axes with the corner, the oracle
$\lambda$, and the discrepancy $\lambda$ all marked, so the flatness of the
error near the minimum is visible as the three markers sitting close together
on the curve's elbow.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

The gate is on the *error* the parameter choice delivers, not on the parameter
itself. Gating on $\lambda$ would fail a method that is working perfectly well,
since the whole finding here is that $\lambda$ is poorly determined while the
error is not.

In [ ]:
validate.check(
    np.all(np.diff(res_sweep) > 0) and np.all(np.diff(sol_sweep) < 0),
    "the L-curve is traversed monotonically, so its corner is well defined",
    f"residual rises and solution norm falls across all {len(lam_sweep)} values "
    "of lambda",
)
validate.check(
    err_corner / err_best < 1.6,
    "the L-curve corner, using only A and b, lands near the oracle's error",
    f"error {err_corner:.4f} at lambda = {lam_corner:.3e} against the oracle's "
    f"{err_best:.4f} at {lam_best:.3e}: a ratio of {err_corner/err_best:.4f}",
)
validate.check(
    err_disc / err_best < 1.6,
    "and so does the discrepancy principle, from the noise level alone",
    f"error {err_disc:.4f} at lambda = {lam_disc:.3e}, a ratio of "
    f"{err_disc/err_best:.4f}",
)
validate.close(
    np.array([res_sweep[i_disc]]), np.array([delta]),
    "the discrepancy choice does match the residual to the noise level",
    # the sweep grid steps lambda by a factor of 1.12, so the nearest grid
    # point can miss the exact discrepancy level by several per cent.
    rtol=1.2e-1, atol=0.0,
)
validate.check(
    abs(np.log10(lam_disc / lam_corner)) > 1.0,
    "even though the two rules disagree about lambda by more than a decade",
    f"corner {lam_corner:.3e} against discrepancy {lam_disc:.3e}, a gap of "
    f"{abs(np.log10(lam_disc/lam_corner)):.3f} decades, for errors differing by "
    f"only {100*abs(err_disc-err_corner)/err_best:.1f}%: the error surface is "
    "flat near its minimum, so recovering lambda is not the goal",
)

## Exercise 8 — Changing the penalty: $\ell^1$, and why it gives exact zeros

Every method so far penalised $\|\mathbf{x}\|_2^2$, and every solution it
returned had all $n$ entries nonzero. That is not an accident of the examples.
{eq}`eq-pinv-filter` multiplies each SVD coefficient by a factor strictly
between 0 and 1, so nothing is ever set exactly to zero. Ridge *shrinks*; it
does not *select*.

Replacing the penalty with $\|\mathbf{x}\|_1$ changes that completely.
[§0.3](../00-machine/vectors-norms-inner-products.ipynb) drew the $\ell^1$ unit
ball as a diamond, and the diamond's corners are exactly the points where
coordinates vanish. Minimising subject to a linear constraint means inflating
the ball until it first touches the solution set, and a diamond touches
a generic line at a corner while a circle touches at a smooth point.

In [ ]:
# (solution hidden on the public site)


{eq}`eq-pinv-lasso` has no closed form, but it has a clean iteration. The
**soft-thresholding** operator

```{math}
:label: eq-pinv-soft
S_\tau(v) = \operatorname{sign}(v)\max(|v| - \tau,\, 0)
```

is the exact minimiser of $\tfrac{1}{2}(z - v)^2 + \tau|z|$ over $z$, and
**ISTA** (iterative shrinkage-thresholding) alternates a gradient step on the
smooth part with one application of it:

```{math}
:label: eq-pinv-ista
\mathbf{x}^{(j+1)} = S_{\lambda/L}\!\left(
  \mathbf{x}^{(j)} + \tfrac{1}{L}A^{\top}
  \bigl(\mathbf{b} - A\mathbf{x}^{(j)}\bigr)\right),
\qquad L = \|A\|_2^2 .
```

Convergence is guaranteed for any step size $1/L$ with $L \ge \|A\|_2^2$
{cite}`beck2009`. The optimality condition for {eq}`eq-pinv-lasso` is a
subgradient statement, and it is exact, which makes it the ideal validation:
writing $\mathbf{g} = A^{\top}(\mathbf{b} - A\mathbf{x})$,

```{math}
:label: eq-pinv-kkt
g_i = \lambda\operatorname{sign}(x_i) \;\text{ where } x_i \ne 0,
\qquad |g_i| \le \lambda \;\text{ where } x_i = 0 .
```

The test problem is a sparse recovery in the compressed-sensing regime: an
$80\times200$ matrix `A_sp = rng_sp.standard_normal((80, 200)) / np.sqrt(80)`
with `rng_sp = np.random.default_rng(1)`, a true signal with exactly 6 nonzeros
at positions `rng_sp.choice(200, size=6, replace=False)` and values
`rng_sp.uniform(1.0, 3.0, 6) * rng_sp.choice([-1.0, 1.0], 6)`, and data
`b_sp = A_sp @ x_sparse + 1e-3 * rng_sp.standard_normal(80)`. The system is
underdetermined by 120, so without a prior there is no hope; with the right
prior there is.

**Part a)** Write `soft(v, tau)` implementing {eq}`eq-pinv-soft` with
`np.sign` and `np.maximum`, and verify it exactly rather than by eye. For 2000
values `v = np.random.default_rng(3).standard_normal(2000) * 3` and
$\tau = 0.7$, confirm that where $z = S_\tau(v) \ne 0$ the stationarity
condition $z - v + \tau\operatorname{sign}(z) = 0$ holds to $10^{-15}$, and
that where $z = 0$ one has $|v| \le \tau$.

**Part b)** Write `ista(A, b, lam, n_iter=2000)` implementing
{eq}`eq-pinv-ista`, starting from `x = np.zeros(A.shape[1])` and using
`L = np.linalg.norm(A, 2) ** 2`. Return the final iterate. Confirm 2000
iterations suffice by comparing against a 20000-iteration run at
$\lambda = 0.05$: the two should agree to $10^{-12}$.

**Write this one yourself** — the implementation is the lesson.

**Part c)** Run it for $\lambda \in \{0.5, 0.2, 0.1, 0.05, 0.02, 0.01,
0.005\}$. For each, report the number of entries with $|x_i| > 10^{-10}$,
whether that support equals the true support exactly, and the relative error.
The first five values all recover the support **exactly** — 6 nonzeros in the
right 6 of 200 positions — while $\lambda \le 0.01$ starts admitting spurious
entries.

**Part d)** Compare against $\ell^2$ at the same $\lambda = 0.05$. Compute the
ridge solution `np.linalg.solve(A_sp.T @ A_sp + 0.05 * np.eye(200), A_sp.T @
b_sp)` and the minimum-norm solution `np.linalg.pinv(A_sp) @ b_sp`. Report the
number of nonzero entries of each (200 in both cases), the smallest absolute
entry of the ridge solution (about $4\times10^{-3}$ — small, but not zero), the
relative error of each (about 0.785), and the fraction of each solution's norm
sitting *off* the true support. For $\ell^1$ that fraction is exactly 0; for
ridge it is about 0.78, so nearly four fifths of the $\ell^2$ answer is in the
wrong places.

**Part e)** Verify optimality with {eq}`eq-pinv-kkt` at $\lambda = 0.05$.
Compute $\mathbf{g} = A^{\top}(\mathbf{b} - A\mathbf{x})$ and check that
$|g_i - \lambda\operatorname{sign}(x_i)| < 10^{-12}$ on the support and that
$\max_{i \notin \text{supp}}|g_i|/\lambda \le 1$. This is an exact
characterisation of the minimiser, so it confirms ISTA converged to the true
solution of {eq}`eq-pinv-lasso` and not merely to somewhere quiet.

```{admonition} With your assistant
:class: tip
ISTA converges at rate $O(1/j)$; FISTA {cite}`beck2009` reaches $O(1/j^2)$ by
extrapolating each iterate along the line through the previous two, at
essentially no extra cost per step. Ask your assistant to write `fista(A, b,
lam, n_iter)` with the momentum sequence $t_{j+1} = (1 + \sqrt{1 + 4t_j^2})/2$.
Then check it yourself, and check it against the mathematics rather than
against ISTA's output: run it at $\lambda = 0.05$ and verify the KKT conditions
{eq}`eq-pinv-kkt` hold for *its* iterate to $10^{-12}$ on the support and
$|g_i| \le \lambda$ off it. Both algorithms minimise the same convex function,
so both must satisfy the same exact condition — and how many iterations each
needs to get there is then a fair comparison. The check is yours.
```

In [ ]:
# (solution hidden on the public site)


### Validation 8

Sparsity is checked as an exact count of exact zeros, not as "small entries",
because the distinction is the entire point of the exercise. Optimality is
checked by the KKT conditions {eq}`eq-pinv-kkt`, which characterise the
minimiser exactly, rather than by watching the objective stop changing.

In [ ]:
validate.check(
    stationarity < 1e-15 and below_tau <= tau_test,
    "soft-thresholding is the exact minimiser of (z-v)^2/2 + tau|z| (Eq. 11)",
    f"stationarity holds to {stationarity:.2e} where z != 0, and every v with "
    f"z = 0 satisfies |v| = {below_tau:.4f} <= tau = {tau_test}",
)
validate.close(
    x_2000, x_ref,
    "2000 ISTA iterations have converged (against 20000) (Eq. 12)",
    rtol=0.0, atol=1e-12,
)
validate.check(
    all(row[2] for row in l1_rows[:5]) and all(row[1] == 6 for row in l1_rows[:5]),
    "l1 recovers the exact 6-element support of 200 for every lambda >= 0.02",
    f"support sizes {[row[1] for row in l1_rows[:5]]} at lambda "
    f"{[row[0] for row in l1_rows[:5]]}, each equal to the true support",
)
validate.check(
    int((np.abs(x_ridge) > 1e-10).sum()) == 200
    and int((np.abs(x_minnorm) > 1e-10).sum()) == 200,
    "while l2 sets nothing to zero: all 200 entries survive (Eq. 7)",
    f"the smallest ridge entry is {np.abs(x_ridge).min():.3e} — small, but not "
    "zero; the filter factors are strictly between 0 and 1, so they shrink "
    "without selecting",
)
validate.check(
    np.linalg.norm(x_l1[off]) < 1e-12
    and np.linalg.norm(x_ridge[off]) / np.linalg.norm(x_ridge) > 0.5,
    "and the l2 answer puts most of its mass in the wrong places",
    f"{np.linalg.norm(x_ridge[off])/np.linalg.norm(x_ridge):.4f} of the ridge "
    f"solution's norm lies off the true support, against exactly "
    f"{np.linalg.norm(x_l1[off]):.1e} for l1",
)
validate.check(
    kkt_on < 1e-12 and kkt_off <= 1.0 + 1e-9,
    "and the ISTA iterate satisfies the exact KKT conditions (Eq. 13)",
    f"max |g - lambda sign(x)| = {kkt_on:.2e} on the support and "
    f"max |g|/lambda = {kkt_off:.6f} off it: this is the true lasso minimiser, "
    "not merely a converged-looking iterate",
)

---
## Notebook summary

**The pseudoinverse exists for every matrix, and is pinned down by four
conditions.** Built from the SVD as $A^{+} = V\Sigma^{+}U^{\top}$
{eq}`eq-pinv-svd` — inverting what is above a cut-off and zeroing what is below
— it satisfied all four Moore–Penrose conditions {eq}`eq-pinv-mp` to
$4\times10^{-15}$ and agreed with `np.linalg.pinv` to $10^{-16}$. Its two
products are the orthogonal projectors of {eq}`eq-pinv-projectors`, with traces
equal to the rank 3 to twelve digits, and $\max|AA^{+} - I| = 0.420$: emphatically
not an inverse.

**It is discontinuous, and `rcond` is where that is decided.** Adding $10^{-12}$
to one entry of the rank-3 matrix moved $\|A^{+}\|_2$ from $0.860$ to
$5.35\times10^{12}$, a factor of $6\times10^{12}$, exactly as $\sigma_4$ crossed
the default cut-off $10^{-15}\sigma_1 = 1.0\times10^{-14}$. With `rcond=1e-10`
pinned, the same perturbations changed the answer by nothing at all.

**When the answer is not unique, $A^{+}$ picks the shortest one.** On a
$3\times6$ system with a 3-dimensional null space, $A^{+}\mathbf{b}$ had norm
$2.0217$ and was shorter than all $10^{4}$ randomly generated exact solutions,
with Pythagoras holding to $10^{-13}$ on every one. The normal equations on the
same system did not raise: they returned an equally valid but strictly longer
solution, whose entire excess is null-space content, and *how* much longer
turned out to differ between this machine and the CI runner, because which
member of the solution family LU lands on is a pivot-order detail.

**Truncation and Tikhonov are one formula with two switches.** At $\kappa = 100$
the filter form {eq}`eq-pinv-filter` matched the matrix form and the
stacked-$QR$ form {eq}`eq-pinv-stacked` to $6\times10^{-13}$; the ridge bias fell
by a factor of 100 for each 100-fold drop in $\lambda$ (measured ratios 99.05,
99.99, 100.00, so it is $O(\lambda)$); $\|\mathbf{x}_\lambda\|$ approached
$\|A^{\top}\mathbf{b}\|/\lambda$ to six figures; and the effective degrees of
freedom ran monotonically from $12.000000000$ to $0.0175$. At $\kappa = 10^{6}$
the agreement broke: the matrix form drifted to $2\times10^{-7}$ while the
stacked form held $7\times10^{-12}$, a factor of 30,000, each staying inside its
own rate ($\kappa^2\varepsilon$ and $\kappa\varepsilon$) exactly as in
[§2.3](least-squares-four-ways.ipynb).

**Ill-posed is a different condition from ill-conditioned.** The $200\times200$
Gaussian blur has only about 100 singular values above $\varepsilon\sigma_1$ —
the exact count is a LAPACK-driver detail, since the values either side of the
threshold are themselves rounding error — so its $\kappa$ is not measurable. `np.linalg.solve` was wrong by $2.9\times10^{15}$ on
$0.1\%$-noisy data — and still wrong by a factor of 629 on *noiseless* data,
because $1/\sigma_{200} = 8.8\times10^{16}$ amplifies rounding error alone. The
discrete Picard ratios {eq}`eq-pinv-picard` stayed below 4.03 for the clean data
and exceeded $10^{12}$ for the noisy, which locates the failure using only $A$
and $\mathbf{b}$ — checked over the range where both quantities are genuine
numbers rather than rounding error, and reported beyond it.

**Regularization repairs it, and the bias is the price.** The best $\lambda$
brought the relative error to $0.136$, an improvement of $2.2\times10^{16}$ — but
that aggregate splits into $0.013$ on the smooth part of the signal and $0.182$
on the box, whose corners a penalty on $\|\mathbf{x}\|_2$ necessarily rounds off,
a factor of 14 between the two regimes. Truncation at its best $k = 39$ did
equally well, and $\sigma_{39}/\sigma_1 = 2.1\times10^{-3}$ sits at the noise
level, exactly where the Picard plot said to stop.

**$\lambda$ can be chosen without the answer.** The L-curve corner
{eq}`eq-pinv-curvature` landed 1.4 decades below the oracle $\lambda$ and cost
17% more error; the discrepancy principle landed 0.6 decades above it and cost
1.2%. Neither used $\mathbf{x}_{\text{true}}$. The two disagreed with each other
about $\lambda$ by two decades and about the error by 16%, which is the useful
form of the result: the minimum is flat, so a parameter-choice rule need not
find $\lambda$ accurately to be worth using.

**A different penalty gives a different kind of answer.** With $\ell^1$ and
ISTA {eq}`eq-pinv-ista`, five values of $\lambda$ recovered the *exact* 6-element
support out of 200 unknowns from 80 measurements, with zero mass off the true
support and the exact KKT conditions {eq}`eq-pinv-kkt` satisfied to $2\times10^{-15}$.
Ridge at the same $\lambda$ left all 200 entries nonzero and put 78% of its norm
in the wrong places.

**Methods introduced.** `np.linalg.pinv` and its `rcond`, the hand-built
$V\Sigma^{+}U^{\top}$, `np.linalg.matrix_rank`, null-space bases from the full
SVD, the truncated-SVD and Tikhonov filters, the stacked $QR$ formulation,
`ecp.linalg.gaussian_blur`, the discrete Picard plot, L-curve curvature by
`np.gradient`, the discrepancy principle, soft-thresholding and ISTA.

## Outlook

- **A penalty that knows about smoothness.** Tikhonov penalised
  $\|\mathbf{x}\|_2$, which is why it rounded the box's corners. Penalising
  $\|L\mathbf{x}\|_2$ for $L$ a finite-difference operator penalises
  *roughness* instead, and the $\ell^1$ version of that (total variation)
  reconstructs edges sharply. Both are solved by the same machinery: the
  generalized SVD replaces the ordinary one, and ISTA needs only a different
  proximal operator.
- **What the filter factors are hiding.** The whole of Exercises 4 through 7
  rested on having the SVD, which for a $200\times200$ matrix is free and for a
  $10^{6}\times10^{6}$ one is unthinkable.
  [§5.4](../05-numerical/stationary-and-cg.ipynb) builds iterative solvers that
  never factorize at all, and there the *iteration count* becomes the
  regularization parameter: stopping early filters the small singular values
  for exactly the reason truncation does.
- **The same algebra, under other names.** Ridge regression, weight decay, and
  Tikhonov regularization are {eq}`eq-pinv-tikhonov` three times over, and the
  effective degrees of freedom $\sum_i f_i$ is the quantity that explains why a
  model with more parameters than data can still generalize.
  [§8.1](../08-learning/learning-as-least-squares.ipynb) returns to this with
  the singular spectra of trained weight matrices.
- **Why the SVD keeps winning.** Three notebooks in a row have ended with the
  SVD succeeding where the alternatives failed.
  [§4.1](../04-svd/svd-geometry.ipynb) stops treating it as a tool and makes it
  the subject, starting from the question of what $\sigma_i$ actually *means*.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()